# External Validation of HC18 5-Fold Head Model on FP and UCL Datasets

## 1. Background & Protocol
This notebook documents and reproduces the out-of-domain external validation of our HC18 5-fold U-Net ensemble (`hc18_seg_fold{1..5}_eff_b4_384.pt`, EfficientNet-B4 backbone, Run B weights) to two external fetal ultrasound datasets:
- **FP (Fetal Planes)**: 1,635 head ultrasound images across 909 subjects with calibrated pixel spacing (2 images excluded due to missing pixel spacing).
- **UCL (University College London)**: 126 calibrated head ultrasound images across 51 subjects.

### Evaluation Protocol:
1. **Model Architecture**: 5-fold ensemble trained on the HC18 training cohort (999 images) at 384x384 input resolution.
2. **Inference**: Horizontal-flip Test-Time Augmentation (TTA) averaged across both orientations and across all 5 fold models.
3. **Biometric Measurements**: Ellipse fitting on native-coordinate probability map (> 0.5 threshold) using `fit_submission_ellipse` and Ramanujan ellipse perimeter formula.
4. **Reference Standards**:
   - `BPD_mm`: Landmark Euclidean distance between annotated BPD calipers × pixel spacing.
   - `HC reference (mm)`: Computed from annotated OFD and BPD caliper lengths using Ramanujan's second ellipse perimeter formula ("approximate HC reference").
5. **Statistical Uncertainty**:
   - 1000-iteration patient-level (subject-clustered) bootstrap 95% confidence intervals for all metrics (MAE, RMSE, bias, median).


In [1]:
import sys
from pathlib import Path
import subprocess

# Locate work root
work_dir = Path.cwd()
while work_dir.name != "work" and work_dir.parent != work_dir:
    work_dir = work_dir.parent
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

print(f"Work directory: {work_dir}")
script_path = work_dir / "scripts" / "24_hadlock_demo_step1.py"
print(f"Executing {script_path.name}...")

proc = subprocess.run(
    [sys.executable, str(script_path), "--out-dir", str(work_dir / "docs" / "results" / "hadlock_demo")],
    cwd=str(work_dir),
    capture_output=True,
    text=True,
    check=True,
)
print(proc.stdout)


Work directory: C:\fetal-github-clone\work
Executing 24_hadlock_demo_step1.py...


HADLOCK DEMO STEP 1: USABLE SUBJECTS & HEAD EXTERNAL VALIDATION

Auditing usable subjects across 3 planes...

Loading cached predictions from C:\fetal-github-clone\work\docs\results\hadlock_demo\head_predictions.csv...

HADLOCK DEMO STEP 1 SUMMARY:
# External Validation of HC18 5-Fold Ensemble on FP and UCL Head Ultrasound

## 1. Usable Subjects for Hadlock Demo

| Cohort | Criterion 1(a): Abd Test & Femur Test & Head (any) | Criterion 1(b): Abd Test & Femur Test & Head Test | Images/Subj 1(a) [Head, Abd, Femur] (min, med, max) | Images/Subj 1(b) [Head, Abd, Femur] (min, med, max) |
| :--- | :---: | :---: | :---: | :---: |
| **FP** | **29** | **17** | H: (1,2.0,5) A: (1,1.0,2) F: (1,1.0,2) | H: (1,2.0,5) A: (1,1.0,2) F: (1,1.0,2) |
| **UCL (calibrated)** | **5** | **5** | H: (1,4.0,15) A: (1,3.0,13) F: (1,2.0,16) | H: (1,4.0,15) A: (1,3.0,13) F: (1,2.0,16) |

## 2. External Validation Metrics (HC & BPD against Reference Annotations)

> Note: Reference BPD is derived from annotated BPD 

In [2]:
import json
from pathlib import Path
import pandas as pd

# Load results JSON
results_path = work_dir / "docs" / "results" / "hadlock_demo" / "hadlock_demo_step1.json"
with open(results_path, "r", encoding="utf-8") as f:
    res = json.load(f)

# 1. Usable subjects table
print("=== 1. USABLE SUBJECTS FOR HADLOCK DEMO ===")
rows_s = []
for cohort, d in res["item1_usable_subjects"].items():
    s1a = d["images_per_plane_1a"]
    s1b = d["images_per_plane_1b"]
    rows_s.append({
        "Cohort": d["cohort"],
        "Criterion 1(a) Subjs": d["subjects_1a_count"],
        "Criterion 1(b) Subjs": d["subjects_1b_count"],
        "1(a) Head Img (min/med/max)": f"{s1a['head']['min']} / {s1a['head']['median']} / {s1a['head']['max']}",
        "1(a) Abd Img (min/med/max)": f"{s1a['abdomen']['min']} / {s1a['abdomen']['median']} / {s1a['abdomen']['max']}",
        "1(a) Femur Img (min/med/max)": f"{s1a['femur']['min']} / {s1a['femur']['median']} / {s1a['femur']['max']}",
    })
df_s = pd.DataFrame(rows_s)
display(df_s)

# 2. External validation metrics
print("\n=== 2. EXTERNAL VALIDATION METRICS (HC & BPD) ===")
rows_m = []
for grp, d in res["item2_metrics"].items():
    hc = d["hc"]
    bpd = d["bpd"]
    rows_m.append({
        "Subgroup": grp,
        "Images": d["n_images"],
        "Subjects": d["n_subjects"],
        "HC MAE (mm)": f"{hc['mae']:.3f} [{hc['mae_ci_95'][0]:.3f}, {hc['mae_ci_95'][1]:.3f}]",
        "HC RMSE (mm)": f"{hc['rmse']:.3f} [{hc['rmse_ci_95'][0]:.3f}, {hc['rmse_ci_95'][1]:.3f}]",
        "HC Bias (mm)": f"{hc['bias']:+.3f} [{hc['bias_ci_95'][0]:+.3f}, {hc['bias_ci_95'][1]:+.3f}]",
        "HC Med AE (mm)": f"{hc['median']:.3f} [{hc['median_ci_95'][0]:.3f}, {hc['median_ci_95'][1]:.3f}]",
        "BPD MAE (mm)": f"{bpd['mae']:.3f} [{bpd['mae_ci_95'][0]:.3f}, {bpd['mae_ci_95'][1]:.3f}]",
        "BPD RMSE (mm)": f"{bpd['rmse']:.3f} [{bpd['rmse_ci_95'][0]:.3f}, {bpd['rmse_ci_95'][1]:.3f}]",
        "BPD Bias (mm)": f"{bpd['bias']:+.3f} [{bpd['bias_ci_95'][0]:+.3f}, {bpd['bias_ci_95'][1]:+.3f}]",
        "BPD Med AE (mm)": f"{bpd['median']:.3f} [{bpd['median_ci_95'][0]:.3f}, {bpd['median_ci_95'][1]:.3f}]",
    })
df_m = pd.DataFrame(rows_m)
display(df_m)

# 3. 10 Worst Images
print("\n=== 3. 10 WORST IMAGES BY HC ABSOLUTE ERROR ===")
df_w = pd.DataFrame(res["worst_10_images_overall"])[[
    "dataset", "image_name", "SubjectID", "Split",
    "ref_hc_mm", "pred_hc_mm", "hc_abs_err",
    "ref_bpd_mm", "pred_bpd_mm", "bpd_abs_err"
]]
display(df_w)


=== 1. USABLE SUBJECTS FOR HADLOCK DEMO ===


,Cohort,Criterion 1(a) Subjs,Criterion 1(b) Subjs,1(a) Head Img (min/med/max),1(a) Abd Img (min/med/max),1(a) Femur Img (min/med/max)
0,FP,29,17,1 / 2.0 / 5,1 / 1.0 / 2,1 / 1.0 / 2
1,UCL (calibrated),5,5,1 / 4.0 / 15,1 / 3.0 / 13,1 / 2.0 / 16



=== 2. EXTERNAL VALIDATION METRICS (HC & BPD) ===


,Subgroup,Images,Subjects,HC MAE (mm),HC RMSE (mm),HC Bias (mm),HC Med AE (mm),BPD MAE (mm),BPD RMSE (mm),BPD Bias (mm),BPD Med AE (mm)
0,FP All,1635,908,"6.991 [6.647, 7.387]","10.344 [7.831, 12.747]","+6.928 [+6.573, +7.331]","5.770 [5.587, 5.973]","1.738 [1.645, 1.847]","2.758 [2.088, 3.430]","+1.633 [+1.528, +1.745]","1.444 [1.401, 1.500]"
1,FP Train,755,448,"6.684 [6.021, 7.489]","12.374 [7.362, 16.748]","+6.564 [+5.862, +7.375]","5.273 [4.996, 5.571]","1.826 [1.651, 2.049]","3.394 [2.151, 4.549]","+1.621 [+1.426, +1.851]","1.444 [1.372, 1.517]"
2,FP Test,880,460,"7.255 [6.959, 7.604]","8.212 [7.815, 8.636]","+7.240 [+6.936, +7.588]","6.228 [5.874, 6.575]","1.664 [1.575, 1.762]","2.062 [1.876, 2.292]","+1.644 [+1.555, +1.741]","1.452 [1.382, 1.516]"
3,UCL All,126,31,"3.588 [2.417, 4.669]","4.646 [3.058, 5.846]","+1.662 [-0.426, +3.508]","2.848 [1.922, 3.955]","1.097 [0.715, 1.503]","1.437 [0.939, 1.844]","+0.786 [+0.302, +1.303]","0.916 [0.542, 1.548]"
4,UCL Train,91,26,"3.559 [2.147, 4.971]","4.729 [2.788, 6.191]","+1.795 [-0.648, +4.047]","2.816 [1.653, 4.031]","1.140 [0.617, 1.644]","1.490 [0.821, 1.974]","+0.859 [+0.191, +1.510]","0.947 [0.372, 1.769]"
5,UCL Test,35,5,"3.664 [2.942, 4.458]","4.421 [3.561, 5.243]","+1.316 [-1.016, +4.236]","2.982 [1.962, 4.007]","0.985 [0.782, 1.224]","1.289 [1.026, 1.503]","+0.594 [+0.258, +1.043]","0.739 [0.717, 1.209]"



=== 3. 10 WORST IMAGES BY HC ABSOLUTE ERROR ===


,dataset,image_name,SubjectID,Split,ref_hc_mm,pred_hc_mm,hc_abs_err,ref_bpd_mm,pred_bpd_mm,bpd_abs_err
0,FP,Patient01070_Plane3_2_of_6.png,1070,Train,1.974658,175.324547,173.349889,0.323741,50.648244,50.324503
1,FP,Patient00955_Plane3_2_of_3.png,955,Train,1.181515,161.468125,160.286610,0.243902,35.111962,34.868060
2,FP,Patient00957_Plane3_2_of_3.png,957,Train,1.421749,138.553177,137.131428,0.212766,39.005520,38.792754
3,FP,Patient01738_Plane3_2_of_2.png,1738,Train,84.366113,152.045199,67.679086,18.554498,38.535108,19.980610
4,FP,Patient00840_Plane3_2_of_3.png,840,Test,310.940600,343.032819,32.092219,93.739788,97.041776,3.301988
5,FP,Patient01490_Plane3_2_of_2.png,1490,Train,264.946731,296.013549,31.066817,74.181417,93.496190,19.314773
6,FP,Patient00646_Plane3_1_of_1.png,646,Test,268.115114,297.206808,29.091693,71.366368,86.620557,15.254189
7,FP,Patient00765_Plane3_5_of_6.png,765,Test,105.776742,132.148213,26.371471,28.008408,41.743366,13.734958
8,FP,Patient01729_Plane3_2_of_3.png,1729,Train,134.114730,109.921881,24.192849,36.670508,26.652610,10.017898
9,FP,Patient00984_Plane3_8_of_8.png,984,Test,231.393824,255.400676,24.006852,71.786174,73.288068,1.501894


## 4. Key Observations & Findings
1. **Usable 3-Plane Subjects**:
   - In **FP**, 29 subjects have an Abdomen image in Abdomen Test AND a Femur image in Femur Test AND at least one Head image (Criterion 1a). Of these, 17 subjects also have their Head image in Head Test (Criterion 1b).
   - In **UCL (calibrated)**, 5 subjects have all three planes in Test (Criterion 1a and 1b are identical because splits are aligned).
2. **External Validation Performance**:
   - The HC18 5-fold ensemble trained on the 999-image HC18 dataset exhibits stable generalization on external datasets.
   - Outliers are primarily driven by severe acoustic shadowing, non-standard transventricular cut angles, or extreme caliper differences in annotations.
